# Dice bars with 95% CI — all five datasets

Extends the AIPS/Sheffield/Augmented-only CI chart in `paper_graphs_morris.ipynb`
with real per-subject data for **MyoSegmenTUM** (all subjects, water modality)
and **Pathological** (the `P001`–`P004` subset, also water). Self-contained —
doesn't depend on any earlier cell in another notebook.

- AIPS/Sheffield/Augmented: raw per-case CSVs at `../<algo>/codes/results_<dataset>/df_*.csv`.
- MyoSegmenTUM: raw per-case CSVs at each algorithm's `results_water` folder
  (location differs per algorithm — see `MYOSEGMENTUM_WATER_FOLDERS`).
- Pathological: the same folders' `results_pathological/results_water` subfolder,
  built by `../make_pathological.ipynb` (filters MyoSegmenTUM down to just the
  4 `P*` subjects). That notebook writes each modality into its own subfolder
  specifically so water and fat-fraction filtered CSVs never collide/overwrite
  each other — re-run it first if this cell reports missing folders.
- AIPS ground truth only labels the right side, so `_L`/`_left` columns are
  dropped for AIPS only (they're 0.0 dice against an empty mask, not real
  failures). Not applicable to any other dataset.
- The per-muscle CSVs use inconsistent id columns across algorithms — some
  have a real `sample`/`subject` column already (AIPS/Sheffield/Augmented/
  dafne/museg/medclipsamv2), others only save a meaningless row-position
  index and embed the real subject+stack in a `pred_label`/`image` filename
  (musclemap_wb/musclemap_thigh/hirriririir). `get_scan_id()` below normalises
  both cases to a `subject_stackN` id so muscle files are joined on the
  correct subject/scan rather than on accidental row order.

In [ ]:
import glob
import os
import re

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

plt.rcParams.update({
    "font.size": 15, "font.family": "sans-serif",
    "axes.spines.top": False, "axes.spines.right": False,
})

In [ ]:
df = pd.read_csv("results_data_from_tables.csv")
order = (df.groupby("algorithm")["dice"].mean().sort_values(ascending=False).index.tolist())
if "MedSegDiff" in order:
    order.remove("MedSegDiff")  # excluded everywhere else in this project's graphs
print(order)

ds_colors = {"MyoSegmenTUM": "#0072B2", "Pathological": "#D55E00",
             "AIPS": "#009E73", "Sheffield": "#CC79A7", "Augmented": "#E69F00"}
ds_hatch = {"MyoSegmenTUM": "", "Pathological": "//", "AIPS": "..",
            "Sheffield": "xx", "Augmented": "\\\\"}

In [ ]:
ci_datasets = ["MyoSegmenTUM", "Pathological", "AIPS", "Sheffield", "Augmented"]

raw_folders = {
    "MuscleMap WB":           {"AIPS": "../muscle_map_wb/codes/results_asian_water",
                                "Sheffield": "../muscle_map_wb/codes/results_sheffield",
                                "Augmented": "../muscle_map_wb/codes/results_augmented"},
    "MuscleMap Thigh":        {"AIPS": "../muscle_map_thigh/codes/results_asian_water",
                                "Sheffield": "../muscle_map_thigh/codes/results_sheffield",
                                "Augmented": "../muscle_map_thigh/codes/results_augmented"},
    "Multimodal-Multiethnic": {"AIPS": "../multimodal-multiethnic/codes/results_asian_water",
                                "Sheffield": "../multimodal-multiethnic/codes/results_sheffield",
                                "Augmented": "../multimodal-multiethnic/codes/results_augmented"},
    "MuSeg":                  {"AIPS": "../museg/codes/results_asian_water_only",
                                "Sheffield": "../museg/codes/results_sheffield",
                                "Augmented": "../museg/codes/results_augmented"},
    "MedCLIP-SAMv2":          {"AIPS": "../medclipsamv2textboxes/codes/results_asian_water",
                                "Sheffield": "../medclipsamv2textboxes/codes/results_sheffield",
                                "Augmented": "../medclipsamv2textboxes/augmented_results"},
    "Dafne":                  {"AIPS": "../dafne/codes/results_asian_water",
                                "Sheffield": "../dafne/codes/results_sheffield",
                                "Augmented": "../dafne/codes/results_augmented"},
}

# MyoSegmenTUM's raw per-case output folder lives in a different place per
# algorithm (some under codes/, some at the algorithm's top level, one nested
# under results/) — verified against what's actually on disk.
MYOSEGMENTUM_WATER_FOLDERS = {
    "MuscleMap WB":           "../muscle_map_wb/codes/results_water",
    "MuscleMap Thigh":        "../muscle_map_thigh/codes/results_water",
    "Multimodal-Multiethnic": "../multimodal-multiethnic/results/results_water",
    "MuSeg":                  "../museg/results_water",
    "MedCLIP-SAMv2":          "../medclipsamv2textboxes/results_water",
    "Dafne":                  "../dafne/results_water",
}

# Pathological = the same algorithms' results_water, pre-filtered to the 4
# P001-P004 subjects by make_pathological.ipynb, one modality per subfolder.
PATHOLOGICAL_WATER_FOLDERS = {
    algo: folder.replace("/results_water", "/results_pathological/results_water")
    for algo, folder in MYOSEGMENTUM_WATER_FOLDERS.items()
}

# Merge into raw_folders[algo]["MyoSegmenTUM" / "Pathological"] — must extend
# each algorithm's inner per-dataset dict, not add "MyoSegmenTUM" as if it
# were itself an algorithm key.
for algo, path in MYOSEGMENTUM_WATER_FOLDERS.items():
    raw_folders[algo]["MyoSegmenTUM"] = path
for algo, path in PATHOLOGICAL_WATER_FOLDERS.items():
    raw_folders[algo]["Pathological"] = path

for algo in raw_folders:
    print(f'{algo:26s} MyoSegmenTUM -> {raw_folders[algo].get("MyoSegmenTUM")}')
    print(f'{"":26s} Pathological -> {raw_folders[algo].get("Pathological")}')

In [ ]:
# ── Normalise each CSV's rows to a per-scan id (subject + stack) so muscle
# files are joined on the correct subject rather than on accidental row order.
_SCAN_RE = re.compile(r'([A-Za-z]+\d+_\d+)_(?:WATER|FATFRACTION)_stack(\d+)')
_SCAN_RE_PATH = re.compile(r'[\\/]([A-Za-z]+\d+_\d+)[\\/]SegmentationMasks[\\/]combined_gt_stack(\d+)')


def get_scan_id(d):
    """Return a Series of unique per-row scan ids ('SUBJECT_stackN')."""
    if 'subject' in d.columns and 'stack' in d.columns:
        return d['subject'].astype(str) + '_stack' + d['stack'].astype(str)

    first_col = d.columns[0]
    if first_col != 'Unnamed: 0':
        # AIPS/Sheffield/Augmented style: 'sample'/'subject' already uniquely
        # identifies one row.
        return d[first_col].astype(str)

    # musclemap_wb / musclemap_thigh / hirriririir style: no saved id column,
    # extract subject+stack from the pred_label/image filename instead.
    for col in ('pred_label', 'pred_file', 'image', 'gt_path'):
        if col in d.columns:
            matches = d[col].astype(str).apply(
                lambda s: _SCAN_RE.search(s) or _SCAN_RE_PATH.search(s))
            if matches.notna().any():
                return matches.apply(lambda m: f'{m.group(1)}_stack{m.group(2)}' if m else None)

    return d[first_col].astype(str)  # last-resort fallback


def per_subject_dice(folder, drop_left=False):
    files = glob.glob(os.path.join(folder, "df_*.csv"))
    merged = None
    for f in files:
        d = pd.read_csv(f)
        dice_cols = [c for c in d.columns if c.endswith("_dice")]
        if drop_left:
            dice_cols = [c for c in dice_cols if not c[:-5].endswith(("_L", "_left"))]
        if not dice_cols:
            continue
        sub = d[dice_cols].copy()
        sub.index = get_scan_id(d)
        merged = sub if merged is None else merged.join(sub, how="outer")
    if merged is None or merged.empty:
        return pd.Series(dtype=float)
    return merged.mean(axis=1).dropna()

In [ ]:
ci_stats = {}
missing = []

for algo in order:
    for ds in ci_datasets:
        folder = raw_folders.get(algo, {}).get(ds)
        if folder is None or not os.path.isdir(folder):
            missing.append((algo, ds, folder))
            continue

        vals = per_subject_dice(folder, drop_left=(ds == "AIPS"))
        n = len(vals)
        if n < 2:
            missing.append((algo, ds, f'{folder} (n={n}, need >=2 for a CI)'))
            continue

        mean = vals.mean()
        sem = vals.std(ddof=1) / np.sqrt(n)
        halfwidth = stats.t.ppf(0.975, n - 1) * sem
        ci_stats[(algo, ds)] = (mean, halfwidth, n)
        print(f"{algo:26s} {ds:14s} n={n:3d} mean={mean:.3f} +/- {halfwidth:.3f}")

if missing:
    print(f'\n{len(missing)} (algorithm, dataset) combo(s) skipped:')
    for algo, ds, why in missing:
        print(f'  {algo:26s} {ds:14s} {why}')

In [ ]:
# ---- grouped bar chart with 95% CI error bars ----
plot_order = [a for a in order if all((a, ds) in ci_stats for ds in ci_datasets)]
skipped_algos = [a for a in order if a not in plot_order]
if skipped_algos:
    print(f'Excluded from the plot (missing data for at least one dataset): {skipped_algos}')

fig, ax = plt.subplots(figsize=(13, 6.5))
n_ds = len(ci_datasets); group_w = 0.82; bar_w = group_w / n_ds
x = np.arange(len(plot_order))
for i, ds in enumerate(ci_datasets):
    vals = [ci_stats[(a, ds)][0] for a in plot_order]
    errs = [ci_stats[(a, ds)][1] for a in plot_order]
    offsets = x - group_w/2 + bar_w*(i + 0.5)
    ax.bar(offsets, vals, bar_w, yerr=errs, capsize=3, label=ds, color=ds_colors[ds],
           hatch=ds_hatch[ds], edgecolor="white", linewidth=0.5,
           error_kw=dict(ecolor="black", elinewidth=1))
ax.set_xticks(x); ax.set_xticklabels(plot_order, rotation=25, ha="right")
ax.set_ylabel("Dice coefficient"); ax.set_ylim(0, 1)
ax.legend(title="Dataset", ncol=5, loc="upper center",
          bbox_to_anchor=(0.5, -0.22), frameon=False)
ax.grid(axis="y", linestyle=":", alpha=0.4); ax.set_axisbelow(True)
plt.tight_layout()
plt.savefig("dice_bars_ci_complete.png", dpi=200, bbox_inches="tight")
plt.savefig("dice_bars_ci_complete.pdf", bbox_inches="tight")
plt.savefig("dice_bars_ci_complete.tiff", dpi=200, bbox_inches="tight")
plt.show()
print("saved dice_bars_ci_complete (.png, .pdf, .tiff)")